## Python Foundations for Data Engineering

#### Why Python for DE specifically?
In Data Engineering, Python is used to:

- Move data — read files, call APIs, connect to databases
- Clean data — fix missing values, wrong types, duplicates
- Transform data — reshape, merge, aggregate datasets
- Automate pipelines — run scripts on a schedule

So every concept we cover will be tied to a real DE use case.

### Lesson 1.1 — Data Types & Structures That Matter in DE

You already know variables. What matters in DE is knowing which structure to use for which job:

In [2]:
# --- LISTS — ordered, for rows of data 
records = ['Kigali', 'Huye', 'Musanze']

# --- DICTIONARIES — key-value pairs, like a single row in a table
person = {
    'name': 'Nadia',
    'district': "Kigali",
    'age': 22
}

# --- LIST OF DICTS — the most common structure in DE (think: a table)
data = [
    {"name": "Alice", "district": "Kigali", "score": 87},
    {"name": "Bob",   "district": "Huye",   "score": 92},
    {"name": "Carol", "district": "Musanze","score": 78},
]

# --- SETS — for removing duplicates fast
districts = {"Kigali", "Huye", "Kigali", "Musanze"}
print(districts) 

# --- TUPLES — immutable, used for fixed config values
DB_CONFIG = ("localhost", 5432, "mydb")

{'Kigali', 'Musanze', 'Huye'}


#### DE rule of thumb:
- Got rows of data? → list of dicts
- Got config/settings? → dict or tuple
- Need to deduplicate? → set

### Lesson 1.2 — Functions: Writing Clean, Reusable Pipeline Code

In DE, every step of a pipeline is a function. Bad DE code is one giant script. Good DE code is small, focused functions.

In [3]:
# Bad — not reusable, hard to debug
data = [{"name": "Alice", "score": None}, {"name": "BOB", "score": 92}]
for row in data:
    if row['score'] is None:
        row['score'] = 0
    row['name'] = row['name'].capitalize()

# Good — each step is its own function
def clean_name(name:str) -> str:
    """Capitalize a name string."""
    return name.strip().capitalize()

def fill_missing_score(score, default: int=0) -> int:
    """Replace None score with a default value."""
    return score if score is not None else default 

def clean_record(record: dict) -> dict:
    """Clean a single data record."""
    return {
        "name": clean_name(record["name"]),
        "score": fill_missing_score(record["score"])
    }

def clean_dataset(data: list) -> list:
    """Apply cleaning to every record in a dataset."""
    return [clean_record(row) for row in data]

# Run it
data = [{"name": "Alice", "score": None}, {"name": "BOB", "score": 92}]
cleaned = clean_dataset(data)
print(cleaned)

[{'name': 'Alice', 'score': 0}, {'name': 'Bob', 'score': 92}]


Notice:
- Type hints (name: str, -> str) — shows what goes in and out
- Docstrings — one line explaining what the function does
- Single responsibility — each function does ONE thing

### Lesson 1.3 — File Handling: Reading Real Data

In [4]:
import csv
import json

# --- Reading a CSV file
def read_csv(filepath: str) -> list:
    """ Read a CSV file and return a list of dicts."""
    with open(filepath, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return [row for row in reader]

# --- Reading a JSON file
def read_json(filepath: str) -> dict:
    """ Read a JSON file and return a dict."""
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)

# --- Writing cleaned data back to CSV
def write_csv(data: list, filepath: str) -> None:
    """ Write a list of dicts to a CSV file."""
    if not data:
        return
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=data[0].keys())
        writer.writeheader()
        writer.writerows(data)
    print(f"Saved {len(data)} records to {filepath}")

The with `open(...)` pattern is important — it automatically closes the file even if something goes wrong.

### Lesson 1.4 — Error Handling: Pipelines Must Not Crash

A pipeline running at 3am on 10,000 rows cannot crash because of one bad row. This is how you handle it:

In [5]:
def safe_convert_to_int(value, default=0):
    """ Safely convert a value to int, return default if it fails."""
    try:
        return int(value)
    except (ValueError, TypeError):
        return default

# Test it
print(safe_convert_to_int("42"))       # 42
print(safe_convert_to_int("abc"))      # 0
print(safe_convert_to_int(None))       # 0
print(safe_convert_to_int("7.5"))      # 0 — "7.5" can't go directly to int

42
0
0
0


For pipelines, also use logging instead of `print()`:


In [6]:
import logging

# Set this up once at the top of your script
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s"
)

def process_record(record):
    try:
        result = clean_record(record)
        logging.info(f"Processed: {record['name']}")
        return result
    except Exception as e:
        logging.error(f"Failed to process record {record}: {e}")
        return None

##  Exercises

1. Create a list of 5 dicts representing districts in Rwanda, each with `name`, `province`, and `population`. Write a function that filters only districts with population above a number you pass in.

In [7]:
from typing import List, Dict

# Sample data
districts = [
    {"name": "Gasabo",   "province": "Kigali City", "population": 600000},
    {"name": "Kicukiro", "province": "Kigali City", "population": 400000},
    {"name": "Nyarugenge","province": "Kigali City", "population": 350000},
    {"name": "Huye",     "province": "Southern",    "population": 330000},
    {"name": "Musanze",  "province": "Northern",    "population": 420000},
]

def filter_by_population(data: List[Dict], min_population: int) -> List[Dict]:
    """ Return districts with population above min_population."""
    return [d for d in data if d.get("population", 0) > min_population]


# Test
result = filter_by_population(districts, 400000)
print(result)

[{'name': 'Gasabo', 'province': 'Kigali City', 'population': 600000}, {'name': 'Musanze', 'province': 'Northern', 'population': 420000}]


- `get("population", 0)` avoids crashes if the key is missing

 2. Write a function `clean_text(text)` that: lowercases the text, strips whitespace, and replaces multiple spaces with a single space. Test it with messy strings.

In [8]:
import re

def clean_text(text: str) -> str:
    """ Normalize text: lowercase, trim, and remove extra spaces."""
    if text is None:
        return ""

    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)  # replace multiple spaces with one
    return text

# Tests
print(clean_text("   Hello       WORLD   "))     # "hello world"
print(clean_text("DATA     engineering"))    # "data engineering"
print(clean_text(None))                      # ""

hello world
data engineering



3. Create a CSV file manually (in Excel or a text editor) with at least 5 rows of any data, then write a Python script that reads it, prints how many rows it has, and saves a copy with `_cleaned` added to the filename.

In [10]:
import csv
import os

def read_csv(filepath: str) -> list:
    """Read CSV into list of dicts."""
    if not os.path.exists(filepath):
        logging.error(f"File not found: {filepath}")
        return []
    with open(filepath, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return [row for row in reader]

def write_csv(data: list, filepath: str) -> None:
    """ Write list of dicts to CSV."""
    if not data:
        return

    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=data[0].keys())
        writer.writeheader()
        writer.writerows(data)

def process_file(filepath: str):
    data = read_csv(filepath)

    print(f"Number of rows: {len(data)}")

    # Create new filename
    base, ext = os.path.splitext(filepath)
    new_filepath = f"{base}_cleaned{ext}"

    write_csv(data, new_filepath)

    print(f"Saved cleaned file as: {new_filepath}")


# Run
process_file("data.csv")

2026-04-17 15:11:06,345 — ERROR — File not found: data.csv


Number of rows: 0
Saved cleaned file as: data_cleaned.csv


4. Write a function that takes a list of dicts and a column name, and returns the list sorted by that column. Handle the case where the column doesn't exist using try/except and logging.

In [11]:
import logging
from typing import List, Dict

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s"
)

def sort_by_column(data: List[Dict], column: str) -> List[Dict]:
    """Sort list of dicts by a given column."""
    try:
        return sorted(data, key=lambda x: x[column])
    except KeyError:
        logging.error(f"Column '{column}' does not exist in data.")
        return data
    except Exception as e:
        logging.error(f"Unexpected error: {e}")
        return data


# Test
data = [
    {"name": "Alice", "score": 87},
    {"name": "Bob", "score": 92},
    {"name": "Carol", "score": 78},
]

sorted_data = sort_by_column(data, "score")
print(sorted_data)

# Trigger error
sort_by_column(data, "age")

2026-04-17 15:20:28,044 — ERROR — Column 'age' does not exist in data.


[{'name': 'Carol', 'score': 78}, {'name': 'Alice', 'score': 87}, {'name': 'Bob', 'score': 92}]


[{'name': 'Alice', 'score': 87},
 {'name': 'Bob', 'score': 92},
 {'name': 'Carol', 'score': 78}]